# Sonda lineal sobre la representación previa al bloque final

El capítulo 5 afirmaba, a partir de la proyección t-SNE de la Figura 5.4, que el
extractor convolucional «ya ha resuelto la tarea» antes de llegar al bloque final. La
objeción es correcta: **t-SNE es una proyección no lineal** y puede producir grupos
visualmente separados aunque los vectores originales no lo estén mediante hiperplanos.

Este notebook mide lo que la figura solo sugiere. Extrae el vector de **4 dimensiones que
entra al bloque final** y ajusta sobre él un clasificador estrictamente lineal (regresión
logística multinomial, sin capas ocultas).

## Diseño

El dataset, la partición y el ruido se construyen **una sola vez** con `DATA_SEED = 42`, y
lo único que varía entre ejecuciones es la **semilla de inicialización de los pesos**. Es
el diseño que permite atribuir la variación a la inicialización y no al pipeline completo;
en el resto de la memoria la semilla mueve las tres cosas a la vez. El orden de los lotes
también se fija, de modo que dos ejecuciones difieren exclusivamente en los pesos
iniciales.

Se miden tres cantidades por ejecución:

1. **Generalización**: la sonda se ajusta sobre las representaciones de *train* y se
   evalúa en *test*. Si es alta, un lector lineal basta y el bloque final —clásico o
   cuántico— no está resolviendo la parte difícil.
2. **Separabilidad**: la sonda se ajusta y se evalúa sobre *test*. Responde a si esos 100
   puntos son linealmente separables en $\mathbb{R}^4$, que es exactamente lo que el
   t-SNE no puede demostrar. Como el conjunto de test es el mismo en las tres
   ejecuciones, la cifra es comparable entre ellas.
3. **Antes y después del circuito**: la misma sonda sobre los tres valores esperados que
   salen de la QNN, para ver si el circuito mejora, conserva o degrada la separabilidad
   que recibe.

MNIST se lee de los ficheros IDX de `data/MNIST/raw/`, sin `torchvision`, replicando lo
que hace `transforms.ToTensor()`: `uint8` dividido por 255 en `float32` con forma
`(1, 28, 28)`.

In [1]:
import os
import struct
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.linear_model import LogisticRegression

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

DATA_SEED = 42                 # datos, particion y ruido: fijos
INIT_SEEDS = [42, 43, 44]      # lo unico que varia

device = torch.device("cpu")
print("device:", device, "| DATA_SEED", DATA_SEED, "| INIT_SEEDS", INIT_SEEDS)

device: cpu | DATA_SEED 42 | INIT_SEEDS [42, 43, 44]


## Datos: construidos una sola vez

In [2]:
RAW = os.path.join(os.path.dirname(os.getcwd()), "data", "MNIST", "raw")
if not os.path.isdir(RAW):
    RAW = os.path.join(os.getcwd(), "data", "MNIST", "raw")
print("MNIST:", RAW)


def read_idx_images(path):
    with open(path, "rb") as fh:
        magic, n, rows, cols = struct.unpack(">IIII", fh.read(16))
        buf = fh.read(n * rows * cols)
    return torch.from_numpy(np.frombuffer(buf, dtype=np.uint8).reshape(n, rows, cols).copy())


def read_idx_labels(path):
    with open(path, "rb") as fh:
        magic, n = struct.unpack(">II", fh.read(8))
        buf = fh.read(n)
    return torch.from_numpy(np.frombuffer(buf, dtype=np.uint8).copy()).long()


class PlainMNIST(Dataset):
    # Equivalente a datasets.MNIST + transforms.ToTensor()
    def __init__(self, images, labels):
        self.data = images
        self.targets = labels

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return self.data[i].to(torch.float32).div(255.0).unsqueeze(0), int(self.targets[i])


X_train = PlainMNIST(read_idx_images(os.path.join(RAW, "train-images-idx3-ubyte"))[:400],
                     read_idx_labels(os.path.join(RAW, "train-labels-idx1-ubyte"))[:400])
X_test = PlainMNIST(read_idx_images(os.path.join(RAW, "t10k-images-idx3-ubyte"))[:100],
                    read_idx_labels(os.path.join(RAW, "t10k-labels-idx1-ubyte"))[:100])

split_generator = torch.Generator()
split_generator.manual_seed(DATA_SEED)
train_subset, val_subset = random_split(X_train, [300, 100], generator=split_generator)
print("train/val/test:", len(train_subset), len(val_subset), len(X_test))

MNIST: C:\Users\SergioHF\Desktop\Mi interés\Computación Quántica\Quanvolutional_NN_classify_reconstruct\data\MNIST\raw
train/val/test: 300 100 100


In [3]:
def gaussian_noise(img, sigma=0.25, generator=None):
    return torch.clamp(img + torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma, 0, 1)


def salt_pepper(img, prob=0.15, generator=None):
    noisy = img.clone()
    mask = torch.rand(img.shape, generator=generator, dtype=img.dtype)
    noisy[mask < prob / 2] = 0
    noisy[mask > 1 - prob / 2] = 1
    return noisy


def speckle(img, sigma=0.35, generator=None):
    return torch.clamp(img + img * torch.randn(img.shape, generator=generator, dtype=img.dtype) * sigma, 0, 1)


def apply_mixed_noise(img, generator):
    r = torch.randint(low=0, high=3, size=(1,), generator=generator).item()
    if r == 0:
        img = speckle(gaussian_noise(img, generator=generator), generator=generator); label = 0
    elif r == 1:
        img = salt_pepper(gaussian_noise(img, generator=generator), generator=generator); label = 1
    else:
        img = speckle(salt_pepper(img, generator=generator), generator=generator); label = 2
    return img, label


class NoisyMNISTDataset(Dataset):
    def __init__(self, mnist_dataset, seed):
        g = torch.Generator(); g.manual_seed(seed)
        self.noisy_images, self.labels = [], []
        for img, _ in mnist_dataset:
            noisy_img, label = apply_mixed_noise(img, generator=g)
            self.noisy_images.append(noisy_img)
            self.labels.append(label)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.noisy_images[idx], self.labels[idx]


train_dataset = NoisyMNISTDataset(train_subset, seed=DATA_SEED)
val_dataset = NoisyMNISTDataset(val_subset, seed=DATA_SEED + 1)
test_dataset = NoisyMNISTDataset(X_test, seed=DATA_SEED + 2)

# El orden de los lotes tambien se fija: dos ejecuciones difieren solo en los pesos.
batch_generator = torch.Generator(); batch_generator.manual_seed(DATA_SEED)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=batch_generator)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("soporte por clase (test):",
      np.bincount(np.asarray(test_dataset.labels, dtype=int), minlength=3))

soporte por clase (test): [31 31 38]


## Modelos: idénticos a los del par gemelo de `XAI/`

In [4]:
estimator = Estimator()
observables = [
    SparsePauliOp.from_list([("ZIII", 1.0), ("IZII", 1.0)]),
    SparsePauliOp.from_list([("ZZII", 1.0)]),
    SparsePauliOp.from_list([("IIZZ", 1.0)]),
]


def create_qnn():
    feature_map = zz_feature_map(4, entanglement="full", reps=1)
    ansatz = real_amplitudes(4, entanglement="reverse_linear", reps=1)
    qc = QuantumCircuit(4)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)
    return EstimatorQNN(circuit=qc, input_params=feature_map.parameters,
                        weight_params=ansatz.parameters, input_gradients=True,
                        estimator=estimator, observables=observables)


class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 4)

    def embed(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return torch.tanh(self.fc2(x)) * np.pi


class ClassicalNet(Backbone):
    def __init__(self):
        super().__init__()
        self.feature_mixer = nn.Sequential(
            nn.Linear(4, 16), nn.Tanh(),
            nn.Linear(16, 16), nn.Tanh(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.feature_mixer(self.embed(x))


class Net(Backbone):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = TorchConnector(qnn)
        self.fc_final = nn.Identity()

    def forward(self, x):
        return self.fc_final(self.qnn(self.embed(x)))

def evaluate(model):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for data, target in test_loader:
            correct += (model(data).argmax(1) == target).sum().item()
            total += target.size(0)
    return correct / total


def train(model):
    """Entrena replicando el bucle de XAI/.

    El original guarda `best_model_state = model.state_dict()` SIN copiar, asi que
    esa referencia sigue los tensores vivos y el `load_state_dict` final deja el
    modelo en la ultima epoca. Se devuelve la accuracy de ese estado final, que es
    el que produjo los resultados de la memoria, y tambien la del estado de la
    mejor epoca de validacion, que es el que la metodologia dice restaurar.
    """
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val, patience, counter = float("inf"), 3, 0
    true_best_state, epochs_run, stopped_early = None, 0, False

    t0 = time.time()
    for epoch in range(10):
        epochs_run += 1
        model.train()
        for data, target in train_loader:
            optimizer.zero_grad()
            criterion(model(data), target).backward()
            optimizer.step()
        model.eval()
        vl, n = 0.0, 0
        with torch.no_grad():
            for data, target in val_loader:
                vl += criterion(model(data), target).item() * data.size(0)
                n += data.size(0)
        vl /= n
        if vl < best_val:
            best_val, counter = vl, 0
            true_best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            counter += 1
            if counter >= patience:
                stopped_early = True
                break
    secs = time.time() - t0

    # Estado final: lo que realmente deja el codigo original.
    acc_final = evaluate(model)

    # Mejor epoca real: lo que la metodologia describe. Se evalua y se descarta,
    # dejando el modelo en el estado final para que la sonda mida ese.
    if true_best_state is None:
        acc_best = acc_final
    else:
        final_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(true_best_state)
        acc_best = evaluate(model)
        model.load_state_dict(final_state)

    return acc_final, acc_best, secs, epochs_run, stopped_early

## Barrido: tres semillas de inicialización

In [5]:
def embeddings(model, loader, post_circuit=False):
    E, Y = [], []
    model.eval()
    with torch.no_grad():
        for data, target in loader:
            z = model.embed(data)
            if post_circuit:
                z = model.qnn(z)
            E.append(z.cpu().numpy()); Y.append(np.asarray(target))
    return np.concatenate(E), np.concatenate(Y)


def probe(Xtr, ytr, Xte, yte):
    gen = LogisticRegression(max_iter=5000).fit(Xtr, ytr).score(Xte, yte)
    sep = LogisticRegression(max_iter=5000).fit(Xte, yte).score(Xte, yte)
    return gen, sep


CACHE = os.path.join(os.path.dirname(os.getcwd()), "outputs")
os.makedirs(CACHE, exist_ok=True)

results, cache = [], {}

for s in INIT_SEEDS:
    torch.manual_seed(s)
    m_cls = ClassicalNet().to(device)
    acc_cls, best_cls, t_cls, ep_cls, es_cls = train(m_cls)

    torch.manual_seed(s)
    m_hyb = Net(create_qnn()).to(device)
    acc_hyb, best_hyb, t_hyb, ep_hyb, es_hyb = train(m_hyb)

    row = {"seed": s,
           "acc_cls": acc_cls, "best_cls": best_cls, "ep_cls": ep_cls, "es_cls": es_cls,
           "acc_hyb": acc_hyb, "best_hyb": best_hyb, "ep_hyb": ep_hyb, "es_hyb": es_hyb}
    for tag, model, post in (("cls", m_cls, False), ("hyb", m_hyb, False), ("post", m_hyb, True)):
        Etr, ytr = embeddings(model, train_loader, post)
        Ete, yte = embeddings(model, test_loader, post)
        cache["%s_%d_tr" % (tag, s)], cache["%s_%d_trY" % (tag, s)] = Etr, ytr
        cache["%s_%d_te" % (tag, s)], cache["%s_%d_teY" % (tag, s)] = Ete, yte
        g, sp = probe(Etr, ytr, Ete, yte)
        row[tag + "_gen"], row[tag + "_sep"] = g, sp
    results.append(row)
    print("seed %d | clasico final %.2f best %.2f (%d ep%s) | hibrido final %.2f best %.2f (%d ep%s)"
          % (s, acc_cls, best_cls, ep_cls, ", early" if es_cls else "",
             acc_hyb, best_hyb, ep_hyb, ", early" if es_hyb else ""), flush=True)
    print("         sonda pre-bloque  cls %.2f/%.2f   hyb %.2f/%.2f   post-QNN %.2f/%.2f"
          % (row["cls_gen"], row["cls_sep"], row["hyb_gen"], row["hyb_sep"],
             row["post_gen"], row["post_sep"]), flush=True)

np.savez(os.path.join(CACHE, "linear_probe_embeddings.npz"), **cache)
print("embeddings guardados en", CACHE)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


seed 42 | clasico final 0.93 best 0.93 (10 ep) | hibrido final 1.00 best 1.00 (10 ep)


         sonda pre-bloque  cls 0.97/0.99   hyb 1.00/1.00   post-QNN 1.00/1.00


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


seed 43 | clasico final 0.81 best 0.94 (10 ep) | hibrido final 0.96 best 0.96 (10 ep)


         sonda pre-bloque  cls 0.98/0.98   hyb 0.97/0.98   post-QNN 0.99/0.98


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


seed 44 | clasico final 1.00 best 1.00 (10 ep) | hibrido final 0.83 best 0.87 (10 ep, early)


         sonda pre-bloque  cls 1.00/1.00   hyb 0.88/0.90   post-QNN 0.94/0.92


embeddings guardados en C:\Users\SergioHF\Desktop\Mi interés\Computación Quántica\Quanvolutional_NN_classify_reconstruct\outputs


## Resumen

In [6]:
def stat(key):
    v = np.array([r[key] for r in results], dtype=float)
    return v.mean(), v.min(), v.max()


print("%-36s %6s %16s" % ("", "media", "rango"))
print("-" * 62)
for label, key in (("modelo clasico (estado final)", "acc_cls"),
                   ("modelo clasico (mejor epoca)", "best_cls"),
                   ("modelo hibrido (estado final)", "acc_hyb"),
                   ("modelo hibrido (mejor epoca)", "best_hyb")):
    m, lo, hi = stat(key)
    print("%-36s %6.2f   [%.2f, %.2f]" % (label, m, lo, hi))
print()
for label, key in (("sonda clasico pre-bloque (4D)", "cls_gen"),
                   ("sonda hibrido pre-circuito (4D)", "hyb_gen"),
                   ("sonda hibrido post-QNN (3D)", "post_gen")):
    m, lo, hi = stat(key)
    print("%-36s %6.2f   [%.2f, %.2f]   train->test" % (label, m, lo, hi))
print()
for label, key in (("separabilidad clasico (4D)", "cls_sep"),
                   ("separabilidad hibrido (4D)", "hyb_sep"),
                   ("separabilidad post-QNN (3D)", "post_sep")):
    m, lo, hi = stat(key)
    print("%-36s %6.2f   [%.2f, %.2f]" % (label, m, lo, hi))

print()
d_cls = np.array([r["cls_gen"] - r["acc_cls"] for r in results])
d_hyb = np.array([r["hyb_gen"] - r["acc_hyb"] for r in results])
print("ventaja de la sonda lineal sobre el modelo completo:")
print("   clasico %+.2f   [%+.2f, %+.2f]" % (d_cls.mean(), d_cls.min(), d_cls.max()))
print("   hibrido %+.2f   [%+.2f, %+.2f]" % (d_hyb.mean(), d_hyb.min(), d_hyb.max()))
print()
print("epocas completadas:",
      [(r["seed"], r["ep_cls"], r["ep_hyb"]) for r in results])

                                      media            rango
--------------------------------------------------------------
modelo clasico (estado final)          0.91   [0.81, 1.00]
modelo clasico (mejor epoca)           0.96   [0.93, 1.00]
modelo hibrido (estado final)          0.93   [0.83, 1.00]
modelo hibrido (mejor epoca)           0.94   [0.87, 1.00]

sonda clasico pre-bloque (4D)          0.98   [0.97, 1.00]   train->test
sonda hibrido pre-circuito (4D)        0.95   [0.88, 1.00]   train->test
sonda hibrido post-QNN (3D)            0.98   [0.94, 1.00]   train->test

separabilidad clasico (4D)             0.99   [0.98, 1.00]
separabilidad hibrido (4D)             0.96   [0.90, 1.00]
separabilidad post-QNN (3D)            0.97   [0.92, 1.00]

ventaja de la sonda lineal sobre el modelo completo:
   clasico +0.07   [+0.00, +0.17]
   hibrido +0.02   [+0.00, +0.05]

epocas completadas: [(42, 10, 10), (43, 10, 10), (44, 10, 10)]
